# Understanding the complete DeepSeek Transformer block

This notebook inspects the pre-RMSNorm block, the dense-prefix/MoE-suffix layout, tensor shapes, and parameter accounting.

In [1]:
import sys
from pathlib import Path

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'src' / 'model').is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError('Could not find the repository root')
sys.path.insert(0, str(project_root))

import torch

from src.experiments.inspect_architecture import (
    count_parameters,
    format_report,
    inspect_architecture,
)
from src.model import DeepSeekConfig, TinyDeepSeek

torch.manual_seed(21)

In [2]:
config = DeepSeekConfig(vocab_size=512, max_seq_len=32)
model = TinyDeepSeek(config)
print(format_report(model))

Complete Tiny-DeepSeek architecture

Layer 0: MLA + Dense SwiGLU | 791,232 parameters
Layer 1: MLA + MoE | 1,972,928 parameters
Layer 2: MLA + MoE | 1,972,928 parameters
Layer 3: MLA + MoE | 1,972,928 parameters
Layer 4: MLA + MoE | 1,972,928 parameters
Layer 5: MLA + MoE | 1,972,928 parameters

Total: 10,918,272


The first layer uses dense SwiGLU. Later layers use DeepSeekMoE. Every layer keeps the same hidden width, so the block boundary is always `[batch, sequence, dim]`.

In [3]:
tokens = torch.randint(0, config.vocab_size, (2, 16))
hidden = model.embed_tokens(tokens)
print('embeddings:', tuple(hidden.shape))

for layer_id, layer in enumerate(model.layers):
    attention_update = layer.attention(layer.attn_norm(hidden))
    after_attention = hidden + attention_update
    ffn_update = layer.ffn(layer.ffn_norm(after_attention))
    hidden = after_attention + ffn_update
    print(f'layer {layer_id}:', tuple(hidden.shape))

logits = model(tokens)
print('logits:', tuple(logits.shape))

embeddings: (2, 16, 256)
layer 0: (2, 16, 256)
layer 1: (2, 16, 256)
layer 2: (2, 16, 256)
layer 3: (2, 16, 256)
layer 4: (2, 16, 256)
layer 5: (2, 16, 256)
logits: (2, 16, 512)


In [4]:
records = inspect_architecture(model)
print('block parameters:', [record['parameters'] for record in records])
print('total parameters:', count_parameters(model))

block parameters: [791232, 1972928, 1972928, 1972928, 1972928, 1972928]
total parameters: 10918272
